# Command line and HTML reports

Two entry points need no Python session: an `onceover` command that
profiles a file from the shell, and a self-contained HTML rendering
of a report for human readers. Both render the same report object
the Python API returns — the claims and numbers are the report's,
read from it, never recomputed.


In [1]:
import json
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import polars as pl

import onceover

names = ["north", "south", "east", "west"]
orders = pl.DataFrame(
    [
        {
            "region_id": i % 4,
            "region": names[i % 4],
            "amount": float((i * 37) % 91),
            "quantity": (i * 13) % 40,
        }
        for i in range(400)
    ]
)
customers = pl.DataFrame(
    {
        "customer_id": list(range(1, 21)),
        "name": [f"customer {i}" for i in range(1, 21)],
    }
)
orders = orders.with_columns(
    customer_id=pl.Series([(i % 20) + 1 for i in range(400)])
)

workdir = Path(tempfile.mkdtemp(prefix="onceover-cli-"))
orders_path = workdir / "orders.parquet"
customers_path = workdir / "customers.csv"
orders.write_parquet(orders_path)
customers.write_csv(customers_path)
print("fixture files:", orders_path.name, customers_path.name)


fixture files: orders.parquet customers.csv


## The command

With no options the command prints the report digest — the same
text `report.to_digest()` returns. `--digest` sets its character
budget:


In [2]:
run = subprocess.run(
    [sys.executable, "-m", "onceover", str(orders_path), "--digest", "700"],
    capture_output=True, text=True, check=True,
)
print(run.stdout)


profile digest: 400 rows x 5 columns, 12 findings, schema 0.7
headline: 400 rows × 5 columns (standard budget); 0 functional dependencies, 2 predictive dependencies, 0 conditional rules
... [8 rolled-up finding groups omitted to fit 700 chars; 12 findings total: column_roles 7, associations 2, dependencies 2, concentration 1; report.select(category="column_roles") retrieves them]



`--format` selects the rendering: `digest` (the default), `json`
(the complete report, identical to the API's), `markdown`, and
`html`. `-o` writes to a file instead of stdout. The remaining flags
are the file-input controls of `profile()`: `--budget`,
`--deadline-seconds`, `--source-row-cap`, `--exact`,
`--sample-seed`, and `--target`. A missing file exits with code 2;
a format the input does not support exits with code 1 and says why.


In [3]:
run = subprocess.run(
    [sys.executable, "-m", "onceover", str(orders_path), "--format", "json"],
    capture_output=True, text=True, check=True,
)
printed = json.loads(run.stdout)
api = onceover.profile(orders)
print("schema:", printed["schema_version"])
print("findings printed by the command:", len(printed["findings"]))
print("findings from the API:", len(api.findings))
assert [f["claim"] for f in printed["findings"]] == [
    f["claim"] for f in api.findings
]


schema: 0.7
findings printed by the command: 12
findings from the API: 12


## The HTML report

`report.to_html()` — also `--format html` on the command line, and
`report.save("report.html", fmt="html")` — renders the report as one
HTML file: the headline, the findings with their categories and
scores beside a chart of findings per category, one card per column
(a five-number strip for a numeric column, the most frequent values
as a bar chart otherwise), and the coverage table.

The charts are Altair / Vega-Lite. The Vega bundle is inlined once
in the page, so the file opens offline and makes no external
requests; each chart adds only its spec. The rendering needs the
`html` extra, which carries Altair and the bundle provider:

```sh
pip install 'onceover[html]'
```

Without it, `to_html()` raises an `ImportError` naming the extra.


In [4]:
report = onceover.profile(orders)
text = report.to_html()
print("report.html bytes:", len(text.encode("utf-8")))
print("external references:", re.findall(r'(?:src|href)="https?:', text))
path = report.save(str(workdir / "report.html"), fmt="html")
print("saved:", Path(path).name)


report.html bytes: 921146
external references: []


saved: report.html


## Several files at once

Given several files, the command reads them as a set of related
tables — table names are the file stems — and prints the
multi-table report of candidate foreign keys, the shell form of
`profile_many`. Only the digest and JSON renderings exist for a
set; asking for markdown or html with several files is an error.


In [5]:
run = subprocess.run(
    [sys.executable, "-m", "onceover", str(orders_path), str(customers_path)],
    capture_output=True, text=True, check=True,
)
print(run.stdout)


Multi-table report: 2 tables, 1 candidate foreign key (1 strong, 0 lower-tier), 0 partial overlaps.
- strong: orders.customer_id -> customers.customer_id (names agree; 20 distinct values)
Considered 20 ordered column pairs across different tables; 8 pairs shared a value family (integer, float, string, or temporal), 4 of those had a unique parent candidate, 3 passed the size prune and were probe-screened, and 0 pairs were large enough to need full verification beyond the probe. Same-table pairs were not examined.

